# Format and save Supplementary Data Tables from raw data files.

Generates all supplementary data tables (see [results](results/)) except Supplementary Table 6, which is generated in [`Upset Plots.ipynb`](scripts/06_figures_tables/UpSet%20Plots.ipynb).

## Notebook Workflow

1. Load cached HyPhy and odds-ratio result objects.
2. Reformat result tables into manuscript supplementary-table structure.
3. Join annotation/conversion metadata and apply final column ordering.
4. Export supplementary tables to spreadsheet files in the results directory.

In [104]:
import os
import sys
import pandas as pd
import numpy as np
from scipy.stats import norm
!{sys.executable} -m pip install openpyxl

# Resolve repo root whether cwd is repo root, scripts/, or a stage subdirectory.
cwd = os.getcwd()
if os.path.basename(cwd) == "orb-selection":
    repo_root = cwd
elif os.path.basename(os.path.dirname(cwd)) == "orb-selection":
    repo_root = os.path.dirname(cwd)
elif os.path.basename(os.path.dirname(os.path.dirname(cwd))) == "orb-selection":
    repo_root = os.path.dirname(os.path.dirname(cwd))
else:
    repo_root = cwd

data = os.path.join(repo_root, "data")
results = os.path.join(repo_root, "results")

src_path = os.path.join(repo_root, "src")
stage03_path = os.path.join(repo_root, "scripts", "03_selection_tests")
stage04_path = os.path.join(repo_root, "scripts", "04_permulation_loss_dup")
for path in (src_path, stage03_path, stage04_path):
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"Using src path: {src_path}")
print(f"Using stage-03 path: {stage03_path}")
print(f"Using stage-04 path: {stage04_path}")

# Import the modules from stage-03/
from hyphy_results_parser import (
    RelaxResult,
    BustedPhResult
)
from id_converter import convert_hogs_to_locs, get_ptep_description

pd.set_option('display.max_columns', None, 'display.max_rows', 20)


Using src path: /Users/calvin/orb-selection/src
Using stage-03 path: /Users/calvin/orb-selection/scripts/03_selection_tests
Using stage-04 path: /Users/calvin/orb-selection/scripts/04_permulation_loss_dup


In [6]:
%load_ext autoreload

In [88]:
def move_column(df, col_name, new_pos):
    cols = list(df.columns)
    cols.insert(new_pos, cols.pop(cols.index(col_name)))
    return df[cols]

## Hyphy Results

In [89]:
hyphy_results = os.path.join(repo_root, "results/hyphy_results_cache/")

# Load the saved RELAX results
relax_result = RelaxResult.load_from_pickle(str(hyphy_results + "relax_results.pkl"))
relax_result_fltrd = relax_result.filter_omega(10000)
relax_hits_df = relax_result_fltrd.get_significant_results()
relax_hits_df.loc['N5.HOG0066983']

p_value                       0.0
k                        0.307001
LRT                     29.059179
MG94xREV_ω_reference     0.479878
MG94xREV_ω_test               0.0
ω1_test                  0.032998
ω1_test_P                0.841624
ω2_test                  0.033758
ω2_test_P                0.133855
ω3_test                  1.137675
ω3_test_P                 0.02452
ω1_ref                   0.000015
ω1_ref_P                 0.841624
ω2_ref                   0.000016
ω2_ref_P                 0.133855
ω3_ref                   1.522192
ω3_ref_P                  0.02452
result                    relaxed
ω_mean_test              0.060187
ω_mean_ref                0.03734
Name: N5.HOG0066983, dtype: object

In [90]:
relax_hits_df = move_column(relax_hits_df, 'result', 0)  # Move 'result' column to the front
relax_hits_df['foreground'] = 'non-orbweavers'
relax_hits_df = move_column(relax_hits_df, 'foreground', 1)  # Move 'foreground' column to the front
relax_hits_df

,result,foreground,p_value,k,LRT,MG94xREV_ω_reference,MG94xREV_ω_test,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref
HOG,,,,,,,,,,,,,,,,,,,,,
N5.HOG0067228,relaxed,non-orbweavers,5.069278e-13,0.031330,52.178060,2.038968e-05,1.110457e+00,0.000000,0.843804,0.389555,0.063292,1.000000,0.092904,0.000000,0.843804,8.541217e-14,0.063292,1.000000,0.092904,0.117560,0.092904
N5.HOG0059719,intensified,non-orbweavers,7.772996e-06,1.108589,19.992907,7.502321e-02,6.447394e-02,0.037770,0.404703,0.046494,0.595297,1.964199,0.000000,0.052061,0.404703,6.279584e-02,0.595297,1.838516,0.000000,0.042963,0.058452
N5.HOG0063256,intensified,non-orbweavers,7.733957e-04,1.136232,11.304017,6.619584e-02,5.929911e-02,0.000000,0.810437,0.086857,0.189563,1.002673,0.000000,0.000000,0.810437,1.164229e-01,0.189563,1.002352,0.000000,0.016465,0.022069
N5.HOG0053197,intensified,non-orbweavers,1.414873e-05,1.212468,18.848973,1.444946e-07,1.613655e-07,0.049339,0.277338,0.065543,0.722662,1.003152,0.000000,0.083597,0.277338,1.056603e-01,0.722662,1.002599,0.000000,0.061049,0.099541
N5.HOG0038544,intensified,non-orbweavers,7.216346e-05,1.097131,15.753260,2.865946e+00,4.260989e-06,0.053668,0.706817,0.054859,0.293183,1.002547,0.000000,0.069531,0.706817,7.093601e-02,0.293183,1.002321,0.000000,0.054018,0.069943
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
N5.HOG0067687,intensified,non-orbweavers,9.106801e-04,1.152606,11.000893,1.822100e-01,1.666336e-01,0.099776,0.262095,0.107294,0.737905,2.158247,0.000000,0.135381,0.262095,1.441878e-01,0.737905,1.949243,0.000000,0.105324,0.141880
N5.HOG0064241,intensified,non-orbweavers,2.168763e-02,1.117584,5.270673,5.236016e-05,4.930010e-01,0.000000,0.302399,0.152434,0.697601,1.862849,0.000000,0.000000,0.302399,1.857939e-01,0.697601,1.744823,0.000000,0.106338,0.129610
N5.HOG0066936,intensified,non-orbweavers,6.469430e-05,1.122134,15.960024,1.096739e-07,8.254460e-01,0.012756,0.285540,0.088955,0.714460,1.001747,0.000000,0.020506,0.285540,1.157564e-01,0.714460,1.001557,0.000000,0.067197,0.088559


In [91]:
# Load the saved BUSTED-PH results
busted_ph_result = BustedPhResult.load_from_pickle(str(hyphy_results + "busted_ph_results.pkl"))
busted_ph_result_fltrd = busted_ph_result.filter_omega(10000)
busted_ph_hits_df = busted_ph_result_fltrd.get_significant_results()
busted_ph_hits_df['foreground'] = 'orbweavers'
busted_ph_hits_df = move_column(busted_ph_hits_df, 'foreground', 0)  # Move 'foreground' column to the front
busted_ph_hits_df.drop(columns=['result'], inplace=True)  # Drop result column
busted_ph_hits_df

,foreground,test_pval,test_LRT,background_pval,background_LRT,shared_pval,shared_LRT,MG94xREV_ω_test,MG94xREV_ω_ref,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref
HOG,,,,,,,,,,,,,,,,,,,,,,,
N5.HOG0058262,orbweavers,1.545704e-03,11.558257,0.194303,1.890383,2.477354e-02,12.855262,0.194701,1.595677e-01,0.000000,0.561941,0.187650,0.437508,41.105141,0.000551,0.000000,0.425298,0.108554,0.561794,1.974842,0.012908,0.104760,0.086476
N5.HOG0059024,orbweavers,1.246802e-03,11.988053,0.500000,0.000000,1.286804e-02,14.473037,0.473107,1.181737e-07,0.000000,0.270546,0.185910,0.702125,3.052891,0.027329,0.064841,0.000000,0.065182,0.898490,1.000000,0.101510,0.213964,0.160075
N5.HOG0062008,orbweavers,4.821284e-08,32.308987,0.408194,0.405731,2.667662e-07,38.749055,0.308877,2.457620e-01,0.000000,0.097881,0.159755,0.897586,48.943871,0.004533,0.062739,0.753940,0.082826,0.189299,1.319449,0.056761,0.365235,0.137874
N5.HOG0032852,orbweavers,1.303768e-11,48.740062,0.477842,0.090656,9.747535e-04,20.573963,0.154262,1.945747e-01,0.009604,0.626402,0.217712,0.372822,3967.329535,0.000775,0.001545,0.069786,0.080352,0.886788,1.120800,0.043426,3.162785,0.120035
N5.HOG0060473,orbweavers,7.599450e-08,31.398915,0.500000,0.000000,4.055495e-04,22.582696,0.239854,2.098764e-01,0.044892,0.586331,0.490225,0.412124,510139.703369,0.001545,0.001943,0.318434,0.109385,0.427393,0.623890,0.254173,788.217417,0.205945
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
N5.HOG0059005,orbweavers,0.000000e+00,205.496307,0.500000,0.000000,1.873424e-07,39.511472,0.316642,2.354017e-01,0.001939,0.451225,0.473936,0.502813,19.647255,0.045962,0.000000,0.602167,0.000579,0.000000,0.555289,0.397833,1.142205,0.220912
N5.HOG0056667,orbweavers,3.393248e-08,33.011492,0.500000,0.000000,5.346662e-06,32.231194,0.157910,1.543083e-01,0.131782,0.991548,0.133366,0.006880,160.277082,0.001573,0.007861,0.328270,0.083886,0.489142,0.452285,0.182587,0.383626,0.126194
N5.HOG0068073,orbweavers,4.157678e-04,14.184473,0.500000,0.000000,1.621171e-03,19.396446,0.231503,2.064418e-01,0.026364,0.018939,0.210999,0.978294,455.127141,0.002767,0.099528,0.855957,0.218815,0.036574,1.000000,0.107469,1.466394,0.200663


In [92]:
# Load the saved BUSTED-PH-rev results
busted_ph_rev_result = BustedPhResult.load_from_pickle(str(hyphy_results + "busted_ph_rev_results.pkl"))
busted_ph_rev_result_fltrd = busted_ph_rev_result.filter_omega(10000)
busted_ph_rev_hits_df = busted_ph_rev_result_fltrd.get_significant_results()
busted_ph_rev_hits_df['foreground'] = 'non-orbweavers'
busted_ph_rev_hits_df = move_column(busted_ph_rev_hits_df, 'foreground', 0)  # Move 'foreground' column to the front
busted_ph_rev_hits_df.drop(columns=['result'], inplace=True)  # Drop result column
busted_ph_rev_hits_df

,foreground,test_pval,test_LRT,background_pval,background_LRT,shared_pval,shared_LRT,MG94xREV_ω_test,MG94xREV_ω_ref,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref
HOG,,,,,,,,,,,,,,,,,,,,,,,
N5.HOG0019712,non-orbweavers,0.000000e+00,809.032492,0.500000,0.000000,0.000000e+00,449.467708,0.000040,0.000017,0.028944,0.645432,0.312607,0.330121,182.484806,0.024447,0.000000,0.071261,0.014606,0.826985,1.000000,0.101753,4.583093,0.113832
N5.HOG0056026,non-orbweavers,8.155119e-10,40.468116,0.500000,0.000000,5.973391e-04,21.698708,0.135780,0.128210,0.051367,0.857685,0.500453,0.139372,5258.822922,0.002943,0.008322,0.719924,0.185418,0.194951,0.859716,0.085125,15.589621,0.115322
N5.HOG0062863,non-orbweavers,3.469447e-14,60.596936,0.500000,0.000000,3.956550e-04,22.638929,0.211636,0.162953,0.081360,0.987843,0.738062,0.007109,4553.459205,0.005048,0.000000,0.068601,0.038564,0.818016,0.530455,0.113383,23.070581,0.091690
N5.HOG0041381,non-orbweavers,0.000000e+00,102.987142,0.109140,3.043957,6.191836e-06,31.909387,0.166545,0.161066,0.000000,0.006265,0.133910,0.987070,25007.129413,0.006665,0.000000,0.381828,0.155732,0.584634,1.691232,0.033538,166.802487,0.147767
N5.HOG0036458,non-orbweavers,2.299926e-02,6.158292,0.133744,2.637359,3.273738e-04,23.069636,0.005095,0.007983,0.001590,0.904070,0.003990,0.095343,4872.117587,0.000587,0.003572,0.994320,0.003753,0.003940,2.572762,0.001740,2.860019,0.008043
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
N5.HOG0067033,non-orbweavers,0.000000e+00,104.322215,0.500000,-20.557023,0.000000e+00,111.260119,0.000032,0.000035,0.000000,0.461302,0.166979,0.534538,178169.075985,0.004160,0.000000,0.485872,0.158264,0.494977,1.948169,0.019151,741.286697,0.115647
N5.HOG0065829,non-orbweavers,0.000000e+00,197.638764,0.500000,-844.964242,2.065015e-11,58.885575,0.000105,0.000077,0.003391,0.582894,0.483803,0.369519,13.393360,0.047587,0.000000,0.282822,0.045675,0.507407,1.848422,0.209771,0.818094,0.410921
N5.HOG0067927,non-orbweavers,0.000000e+00,146.664449,0.500000,0.000000,4.634104e-11,57.183980,0.192987,0.207552,0.035514,0.893746,0.701318,0.089573,6248.594546,0.016681,0.000000,0.030754,0.031915,0.809310,0.728830,0.159936,104.326700,0.142395


In [93]:
all_busted_hits_df = pd.concat([busted_ph_hits_df, busted_ph_rev_hits_df])
all_busted_hits_df

,foreground,test_pval,test_LRT,background_pval,background_LRT,shared_pval,shared_LRT,MG94xREV_ω_test,MG94xREV_ω_ref,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref
HOG,,,,,,,,,,,,,,,,,,,,,,,
N5.HOG0058262,orbweavers,1.545704e-03,11.558257,0.194303,1.890383,2.477354e-02,12.855262,0.194701,1.595677e-01,0.000000,0.561941,0.187650,0.437508,41.105141,0.000551,0.000000,0.425298,0.108554,0.561794,1.974842,0.012908,0.104760,0.086476
N5.HOG0059024,orbweavers,1.246802e-03,11.988053,0.500000,0.000000,1.286804e-02,14.473037,0.473107,1.181737e-07,0.000000,0.270546,0.185910,0.702125,3.052891,0.027329,0.064841,0.000000,0.065182,0.898490,1.000000,0.101510,0.213964,0.160075
N5.HOG0062008,orbweavers,4.821284e-08,32.308987,0.408194,0.405731,2.667662e-07,38.749055,0.308877,2.457620e-01,0.000000,0.097881,0.159755,0.897586,48.943871,0.004533,0.062739,0.753940,0.082826,0.189299,1.319449,0.056761,0.365235,0.137874
N5.HOG0032852,orbweavers,1.303768e-11,48.740062,0.477842,0.090656,9.747535e-04,20.573963,0.154262,1.945747e-01,0.009604,0.626402,0.217712,0.372822,3967.329535,0.000775,0.001545,0.069786,0.080352,0.886788,1.120800,0.043426,3.162785,0.120035
N5.HOG0060473,orbweavers,7.599450e-08,31.398915,0.500000,0.000000,4.055495e-04,22.582696,0.239854,2.098764e-01,0.044892,0.586331,0.490225,0.412124,510139.703369,0.001545,0.001943,0.318434,0.109385,0.427393,0.623890,0.254173,788.217417,0.205945
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
N5.HOG0067033,non-orbweavers,0.000000e+00,104.322215,0.500000,-20.557023,0.000000e+00,111.260119,0.000032,3.549836e-05,0.000000,0.461302,0.166979,0.534538,178169.075985,0.004160,0.000000,0.485872,0.158264,0.494977,1.948169,0.019151,741.286697,0.115647
N5.HOG0065829,non-orbweavers,0.000000e+00,197.638764,0.500000,-844.964242,2.065015e-11,58.885575,0.000105,7.708236e-05,0.003391,0.582894,0.483803,0.369519,13.393360,0.047587,0.000000,0.282822,0.045675,0.507407,1.848422,0.209771,0.818094,0.410921
N5.HOG0067927,non-orbweavers,0.000000e+00,146.664449,0.500000,0.000000,4.634104e-11,57.183980,0.192987,2.075518e-01,0.035514,0.893746,0.701318,0.089573,6248.594546,0.016681,0.000000,0.030754,0.031915,0.809310,0.728830,0.159936,104.326700,0.142395


In [95]:
busted_hits_locs = convert_hogs_to_locs(all_busted_hits_df, os.path.join(repo_root, "data/N5.tsv"), one_random_gene=True)
busted_hits_locs_fltrd = busted_hits_locs.drop(columns=['udiv_genes', 'dmel_orthologs', 'txpt']).drop_duplicates().reset_index(drop=True)
busted_hits_locs_fltrd = move_column(busted_hits_locs_fltrd, 'HOG', 0)
busted_hits_locs_fltrd.rename(columns={
    'LOC': 'U. diversus gene ID',
    'GO_terms': 'U. diversus GO terms',
    'Description': 'U. diversus gene description'
    }, inplace=True)
busted_hits_locs_fltrd

Processing HOGs: 100%|██████████| 292/292 [00:00<00:00, 23452.51it/s]


,HOG,foreground,test_pval,test_LRT,background_pval,background_LRT,shared_pval,shared_LRT,MG94xREV_ω_test,MG94xREV_ω_ref,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref,U. diversus gene ID,U. diversus GO terms,U. diversus gene description
0,N5.HOG0058262,orbweavers,1.545704e-03,11.558257,0.194303,1.890383,2.477354e-02,12.855262,0.194701,1.595677e-01,0.000000,0.561941,0.187650,0.437508,41.105141,0.000551,0.000000,0.425298,0.108554,0.561794,1.974842,0.012908,0.104760,0.086476,129215997,GO:0006470 [Name: protein dephosphorylation]; ...,TGF-beta-activated kinase 1 and MAP3K7-binding...
1,N5.HOG0059024,orbweavers,1.246802e-03,11.988053,0.500000,0.000000,1.286804e-02,14.473037,0.473107,1.181737e-07,0.000000,0.270546,0.185910,0.702125,3.052891,0.027329,0.064841,0.000000,0.065182,0.898490,1.000000,0.101510,0.213964,0.160075,129220007,GO:0045944 [Name: positive regulation of trans...,RING finger protein 10-like
2,N5.HOG0062008,orbweavers,4.821284e-08,32.308987,0.408194,0.405731,2.667662e-07,38.749055,0.308877,2.457620e-01,0.000000,0.097881,0.159755,0.897586,48.943871,0.004533,0.062739,0.753940,0.082826,0.189299,1.319449,0.056761,0.365235,0.137874,129222444,GO:0001405 [Name: presequence translocase-asso...,"grpE protein homolog, mitochondrial Roe1"
3,N5.HOG0032852,orbweavers,1.303768e-11,48.740062,0.477842,0.090656,9.747535e-04,20.573963,0.154262,1.945747e-01,0.009604,0.626402,0.217712,0.372822,3967.329535,0.000775,0.001545,0.069786,0.080352,0.886788,1.120800,0.043426,3.162785,0.120035,129218644,GO:0000407 [Name: pre-autophagosomal structure...,autophagy-related protein 13-like
4,N5.HOG0060473,orbweavers,7.599450e-08,31.398915,0.500000,0.000000,4.055495e-04,22.582696,0.239854,2.098764e-01,0.044892,0.586331,0.490225,0.412124,510139.703369,0.001545,0.001943,0.318434,0.109385,0.427393,0.623890,0.254173,788.217417,0.205945,129231803,GO:0005515 [Name: protein binding],leucine-rich repeat-containing protein 59-like
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287,N5.HOG0067033,non-orbweavers,0.000000e+00,104.322215,0.500000,-20.557023,0.000000e+00,111.260119,0.000032,3.549836e-05,0.000000,0.461302,0.166979,0.534538,178169.075985,0.004160,0.000000,0.485872,0.158264,0.494977,1.948169,0.019151,741.286697,0.115647,129225838,GO:0042273 [Name: ribosomal large subunit biog...,nucleolar complex protein 2 homolog
288,N5.HOG0065829,non-orbweavers,0.000000e+00,197.638764,0.500000,-844.964242,2.065015e-11,58.885575,0.000105,7.708236e-05,0.003391,0.582894,0.483803,0.369519,13.393360,0.047587,0.000000,0.282822,0.045675,0.507407,1.848422,0.209771,0.818094,0.410921,NaN,NaN,NaN
289,N5.HOG0067927,non-orbweavers,0.000000e+00,146.664449,0.500000,0.000000,4.634104e-11,57.183980,0.192987,2.075518e-01,0.035514,0.893746,0.701318,0.089573,6248.594546,0.016681,0.000000,0.030754,0.031915,0.809310,0.728830,0.159936,104.326700,0.142395,129223042,GO:0000981 [Name: RNA polymerase II transcript...,uncharacterized LOC129223042
290,N5.HOG0063651,non-orbweavers,2.807126e-05,19.575234,0.500000,0.000000,1.631340e-05,29.777420,0.000025,3.369798e-05,0.022216,0.824099,0.370172,0.170712,21.378347,0.005190,0.000000,0.540813,0.212053,0.376480,0.853517,0.082707,0.192446,0.150425,129220117,-,uncharacterized LOC129220117


In [96]:
busted_df = get_ptep_description(busted_hits_locs_fltrd)
busted_df

,HOG,foreground,test_pval,test_LRT,background_pval,background_LRT,shared_pval,shared_LRT,MG94xREV_ω_test,MG94xREV_ω_ref,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref,U. diversus gene ID,U. diversus GO terms,U. diversus gene description,P. tepidariorum best BLAST hit for the HOG
0,N5.HOG0058262,orbweavers,1.545704e-03,11.558257,0.194303,1.890383,2.477354e-02,12.855262,0.194701,1.595677e-01,0.000000,0.561941,0.187650,0.437508,41.105141,0.000551,0.000000,0.425298,0.108554,0.561794,1.974842,0.012908,0.104760,0.086476,129215997,GO:0006470 [Name: protein dephosphorylation]; ...,TGF-beta-activated kinase 1 and MAP3K7-binding...,TGF-beta-activated kinase 1 and MAP3K7-binding...
1,N5.HOG0059024,orbweavers,1.246802e-03,11.988053,0.500000,0.000000,1.286804e-02,14.473037,0.473107,1.181737e-07,0.000000,0.270546,0.185910,0.702125,3.052891,0.027329,0.064841,0.000000,0.065182,0.898490,1.000000,0.101510,0.213964,0.160075,129220007,GO:0045944 [Name: positive regulation of trans...,RING finger protein 10-like,E3 ubiquitin-protein ligase RNF10 isoform X1
2,N5.HOG0062008,orbweavers,4.821284e-08,32.308987,0.408194,0.405731,2.667662e-07,38.749055,0.308877,2.457620e-01,0.000000,0.097881,0.159755,0.897586,48.943871,0.004533,0.062739,0.753940,0.082826,0.189299,1.319449,0.056761,0.365235,0.137874,129222444,GO:0001405 [Name: presequence translocase-asso...,"grpE protein homolog, mitochondrial Roe1","grpE protein homolog, mitochondrial"
3,N5.HOG0032852,orbweavers,1.303768e-11,48.740062,0.477842,0.090656,9.747535e-04,20.573963,0.154262,1.945747e-01,0.009604,0.626402,0.217712,0.372822,3967.329535,0.000775,0.001545,0.069786,0.080352,0.886788,1.120800,0.043426,3.162785,0.120035,129218644,GO:0000407 [Name: pre-autophagosomal structure...,autophagy-related protein 13-like,autophagy-related protein 13 homolog
4,N5.HOG0060473,orbweavers,7.599450e-08,31.398915,0.500000,0.000000,4.055495e-04,22.582696,0.239854,2.098764e-01,0.044892,0.586331,0.490225,0.412124,510139.703369,0.001545,0.001943,0.318434,0.109385,0.427393,0.623890,0.254173,788.217417,0.205945,129231803,GO:0005515 [Name: protein binding],leucine-rich repeat-containing protein 59-like,leucine-rich repeat-containing protein 59 isof...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287,N5.HOG0067033,non-orbweavers,0.000000e+00,104.322215,0.500000,-20.557023,0.000000e+00,111.260119,0.000032,3.549836e-05,0.000000,0.461302,0.166979,0.534538,178169.075985,0.004160,0.000000,0.485872,0.158264,0.494977,1.948169,0.019151,741.286697,0.115647,129225838,GO:0042273 [Name: ribosomal large subunit biog...,nucleolar complex protein 2 homolog,nucleolar complex protein 2 homolog
288,N5.HOG0065829,non-orbweavers,0.000000e+00,197.638764,0.500000,-844.964242,2.065015e-11,58.885575,0.000105,7.708236e-05,0.003391,0.582894,0.483803,0.369519,13.393360,0.047587,0.000000,0.282822,0.045675,0.507407,1.848422,0.209771,0.818094,0.410921,NaN,NaN,NaN,uncharacterized protein
289,N5.HOG0067927,non-orbweavers,0.000000e+00,146.664449,0.500000,0.000000,4.634104e-11,57.183980,0.192987,2.075518e-01,0.035514,0.893746,0.701318,0.089573,6248.594546,0.016681,0.000000,0.030754,0.031915,0.809310,0.728830,0.159936,104.326700,0.142395,129223042,GO:0000981 [Name: RNA polymerase II transcript...,uncharacterized LOC129223042,T-cell acute lymphocytic leukemia protein 1 ho...
290,N5.HOG0063651,non-orbweavers,2.807126e-05,19.575234,0.500000,0.000000,1.631340e-05,29.777420,0.000025,3.369798e-05,0.022216,0.824099,0.370172,0.170712,21.378347,0.005190,0.000000,0.540813,0.212053,0.376480,0.853517,0.082707,0.192446,0.150425,129220117,-,uncharacterized LOC129220117,uncharacterized protein


In [98]:
relax_hits_locs = convert_hogs_to_locs(relax_hits_df, os.path.join(repo_root, "data/N5.tsv"), one_random_gene=True)
relax_hits_locs_fltrd = relax_hits_locs.drop(columns=['udiv_genes', 'dmel_orthologs', 'txpt']).drop_duplicates().reset_index(drop=True)
relax_hits_locs_fltrd = move_column(relax_hits_locs_fltrd, 'HOG', 0)
relax_hits_locs_fltrd.rename(columns={
    'LOC': 'U. diversus gene ID',
    'GO_terms': 'U. diversus GO terms',
    'Description': 'U. diversus gene description'
    }, inplace=True)
relax_df = get_ptep_description(relax_hits_locs_fltrd)
relax_df

Processing HOGs: 100%|██████████| 1429/1429 [00:00<00:00, 57345.72it/s]


,HOG,result,foreground,p_value,k,LRT,MG94xREV_ω_reference,MG94xREV_ω_test,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref,U. diversus gene ID,U. diversus GO terms,U. diversus gene description,P. tepidariorum best BLAST hit for the HOG
0,N5.HOG0067228,relaxed,non-orbweavers,5.069278e-13,0.031330,52.178060,2.038968e-05,1.110457e+00,0.000000,0.843804,0.389555,0.063292,1.000000,0.092904,0.000000,0.843804,8.541217e-14,0.063292,1.000000,0.092904,0.117560,0.092904,129223414,GO:0005737 [Name: cytoplasm]; GO:0016925 [Name...,activator of SUMO 1,SUMO-activating enzyme subunit 1
1,N5.HOG0059719,intensified,non-orbweavers,7.772996e-06,1.108589,19.992907,7.502321e-02,6.447394e-02,0.037770,0.404703,0.046494,0.595297,1.964199,0.000000,0.052061,0.404703,6.279584e-02,0.595297,1.838516,0.000000,0.042963,0.058452,129227349,GO:0005737 [Name: cytoplasm]; GO:0042803 [Name...,serine hydroxymethyltransferase-like,serine hydroxymethyltransferase
2,N5.HOG0063256,intensified,non-orbweavers,7.733957e-04,1.136232,11.304017,6.619584e-02,5.929911e-02,0.000000,0.810437,0.086857,0.189563,1.002673,0.000000,0.000000,0.810437,1.164229e-01,0.189563,1.002352,0.000000,0.016465,0.022069,129224590,GO:0003924 [Name: GTPase activity]; GO:0005737...,GPN-loop GTPase 2-like,GPN-loop GTPase 2 isoform X2
3,N5.HOG0053197,intensified,non-orbweavers,1.414873e-05,1.212468,18.848973,1.444946e-07,1.613655e-07,0.049339,0.277338,0.065543,0.722662,1.003152,0.000000,0.083597,0.277338,1.056603e-01,0.722662,1.002599,0.000000,0.061049,0.099541,129220170,GO:0005634 [Name: nucleus]; GO:0032502 [Name: ...,molecular chaperone MKKS-like,chromatin accessibility complex protein 1
4,N5.HOG0038544,intensified,non-orbweavers,7.216346e-05,1.097131,15.753260,2.865946e+00,4.260989e-06,0.053668,0.706817,0.054859,0.293183,1.002547,0.000000,0.069531,0.706817,7.093601e-02,0.293183,1.002321,0.000000,0.054018,0.069943,NaN,NaN,NaN,frizzled-4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1424,N5.HOG0067687,intensified,non-orbweavers,9.106801e-04,1.152606,11.000893,1.822100e-01,1.666336e-01,0.099776,0.262095,0.107294,0.737905,2.158247,0.000000,0.135381,0.262095,1.441878e-01,0.737905,1.949243,0.000000,0.105324,0.141880,NaN,NaN,NaN,trithorax group protein osa
1425,N5.HOG0064241,intensified,non-orbweavers,2.168763e-02,1.117584,5.270673,5.236016e-05,4.930010e-01,0.000000,0.302399,0.152434,0.697601,1.862849,0.000000,0.000000,0.302399,1.857939e-01,0.697601,1.744823,0.000000,0.106338,0.129610,129227186,"GO:0006352 [Name: DNA-templated transcription,...",TATA box-binding protein-like 1,TATA box-binding protein-like 1
1426,N5.HOG0066936,intensified,non-orbweavers,6.469430e-05,1.122134,15.960024,1.096739e-07,8.254460e-01,0.012756,0.285540,0.088955,0.714460,1.001747,0.000000,0.020506,0.285540,1.157564e-01,0.714460,1.001557,0.000000,0.067197,0.088559,129234669,GO:0009058 [Name: biosynthetic process]; GO:00...,eukaryotic translation initiation factor 2B su...,translation initiation factor eIF2B subunit gamma
1427,N5.HOG0060161,intensified,non-orbweavers,1.160829e-03,1.145080,10.551673,5.080422e-02,6.864197e-02,0.000000,0.876649,1.000000,0.118220,2318.115008,0.005132,0.000000,0.876649,1.000000e+00,0.118220,868.514944,0.005132,12.013916,4.575112,129226064,GO:0005515 [Name: protein binding]; GO:0007098...,partitioning defective protein 6,partitioning defective 6 homolog gamma


In [99]:
relax_df.to_excel(os.path.join(repo_root, "results/Supplementary_Table_3_RELAX_hits.xlsx"), index=False)
relax_df.to_excel(os.path.join(repo_root, "results/Supplementary_Table_4_BUSTED_hits.xlsx"), index=False)

## Permutation test results

Reformat results table

In [61]:
%autoreload 2
perm_test_results_dir = os.path.join(repo_root, "results/odds_ratio_test/Results_Apr16/Run1_occ_30-88_10000x_all_orb/")
perm_results = pd.read_csv(os.path.join(perm_test_results_dir, "fltrd_hits.csv"), index_col="HOG")

perm_results["Result"] = np.where(perm_results["Significant by permulation"].str.contains("loss_fg"), "Loss more likely in orb-weavers", None)
perm_results["Result"] = np.where(perm_results["Significant by permulation"].str.contains("loss_bg"), "Loss more likely in non-orb-weavers", perm_results["Result"])
perm_results["Result"] = np.where(perm_results["Significant by permulation"].str.contains("dup_fg"), "Duplication more likely in orb-weavers", perm_results["Result"])
perm_results["Result"] = np.where(perm_results["Significant by permulation"].str.contains("dup_bg"), "Duplication more likely in non-orb-weavers", perm_results["Result"])
perm_results["Result"] = np.where(perm_results["Significant by permulation"].str.contains("loss_fg, dup_bg"), "Loss more likely in orb-weavers, duplication more likely in non-orb-weavers", perm_results["Result"])
perm_results["Result"] = np.where(perm_results["Significant by permulation"].str.contains("loss_bg, dup_fg"), "Loss more likely in non-orb-weavers, duplication more likely in orb-weavers", perm_results["Result"])

perm_results = move_column(perm_results, "Result", 0)  # Move 'Result' column to the front
perm_results["LOR outside permuted avg thresholds"] = np.where(perm_results["Significant by avgd thresholds"].notnull(), "Yes", "No")
perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_fg, dup_bg") & perm_results["Significant by avgd thresholds"].str.contains("loss_fg")), "Yes for loss; no for duplication", perm_results["LOR outside permuted avg thresholds"])
perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_fg, dup_bg") & perm_results["Significant by avgd thresholds"].str.contains("dup_bg")), "No for loss; yes for duplication", perm_results["LOR outside permuted avg thresholds"])
perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_fg, dup_bg") & perm_results["Significant by avgd thresholds"].str.contains("loss_fg, dup_bg")), "Yes for both tests", perm_results["LOR outside permuted avg thresholds"])

perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_bg, dup_fg") & perm_results["Significant by avgd thresholds"].str.contains("loss_bg")), "Yes for loss; no for duplication", perm_results["LOR outside permuted avg thresholds"])
perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_bg, dup_fg") & perm_results["Significant by avgd thresholds"].str.contains("dup_fg")), "No for loss; yes for duplication", perm_results["LOR outside permuted avg thresholds"])
perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_bg, dup_fg") & perm_results["Significant by avgd thresholds"].str.contains("loss_bg, dup_fg")), "Yes for both tests", perm_results["LOR outside permuted avg thresholds"])


perm_results = move_column(perm_results, "LOR outside permuted avg thresholds", 4)  # Move 'LOR outside permuted avg thresholds' column to the front
perm_results = perm_results.drop(columns=["Significant by avgd thresholds", "Significant by permulation"])
perm_results = perm_results.rename(columns={
    "P-value loss more likely in fg": "P-value, loss more likely in orb-weavers", 
    "P-value loss more likely in bg": "P-value, loss more likely in non-orb-weavers", 
    "P-value dup more likely in fg": "P-value, duplication more likely in orb-weavers", 
    "P-value dup more likely in bg": "P-value, duplication more likely in non-orb-weavers", 
    })
perm_results

,Result,Occupancy,Log odds ratio of loss,Log odds ratio of duplication,LOR outside permuted avg thresholds,"P-value, loss more likely in orb-weavers","P-value, loss more likely in non-orb-weavers",P-value duplication more likely in fg,P-value duplication more likely in bg
HOG,,,,,,,,,
N5.HOG0001041,Loss more likely in orb-weavers,46,2.288042,-0.281735,No,0.0038,0.9962,0.5328,0.4672
N5.HOG0001609,Loss more likely in non-orb-weavers,49,-2.841587,2.391485,No,0.9605,0.0395,0.0747,0.9253
N5.HOG0001627,"Loss more likely in orb-weavers, duplication m...",33,2.048043,-4.020457,No for loss; yes for duplication,0.0449,0.9551,0.9745,0.0253
N5.HOG0002031,Loss more likely in non-orb-weavers,46,-3.823601,1.854638,Yes,0.9996,0.0004,0.3311,0.6686
N5.HOG0002072,Loss more likely in orb-weavers,52,3.473733,-0.762726,Yes,0.0108,0.9892,0.7989,0.2011
...,...,...,...,...,...,...,...,...,...
N5.HOG0073114,Loss more likely in orb-weavers,36,1.310443,-1.206534,No,0.0218,0.9782,0.7961,0.2038
N5.HOG0073157,Loss more likely in non-orb-weavers,34,-3.186391,1.988284,Yes,0.9576,0.0424,0.1151,0.8846
N5.HOG0073180,Loss more likely in non-orb-weavers,32,-1.038058,0.724732,No,0.9929,0.0071,0.1394,0.8606


In [83]:
%autoreload 2
perm_locs = convert_hogs_to_locs(perm_results, os.path.join(repo_root, "data/N5.tsv"), one_random_gene=True)
perm_locs_filtered = perm_locs.drop(columns=['udiv_genes', 'dmel_orthologs', 'txpt']).drop_duplicates().reset_index(drop=True)
perm_locs_filtered = move_column(perm_locs_filtered, 'HOG', 0)
perm_locs_filtered.rename(columns={
    'LOC': 'U. diversus gene ID',
    'GO_terms': 'U. diversus GO terms',
    'Description': 'U. diversus gene description'
    }, inplace=True)
perm_locs_filtered

Processing HOGs: 100%|██████████| 1432/1432 [00:00<00:00, 62884.04it/s]


,HOG,Result,Occupancy,Log odds ratio of loss,Log odds ratio of duplication,LOR outside permuted avg thresholds,"P-value, loss more likely in orb-weavers","P-value, loss more likely in non-orb-weavers",P-value duplication more likely in fg,P-value duplication more likely in bg,U. diversus gene ID,U. diversus GO terms,U. diversus gene description
0,N5.HOG0001041,Loss more likely in orb-weavers,46,2.288042,-0.281735,No,0.0038,0.9962,0.5328,0.4672,NaN,NaN,NaN
1,N5.HOG0001609,Loss more likely in non-orb-weavers,49,-2.841587,2.391485,No,0.9605,0.0395,0.0747,0.9253,NaN,NaN,NaN
2,N5.HOG0001627,"Loss more likely in orb-weavers, duplication m...",33,2.048043,-4.020457,No for loss; yes for duplication,0.0449,0.9551,0.9745,0.0253,NaN,NaN,NaN
3,N5.HOG0002031,Loss more likely in non-orb-weavers,46,-3.823601,1.854638,Yes,0.9996,0.0004,0.3311,0.6686,129233975,GO:0008010 [Name: structural constituent of ch...,cuticle protein 10.9-like
4,N5.HOG0002072,Loss more likely in orb-weavers,52,3.473733,-0.762726,Yes,0.0108,0.9892,0.7989,0.2011,129217362,GO:0008010 [Name: structural constituent of ch...,proteoglycan 4-like
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1427,N5.HOG0073114,Loss more likely in orb-weavers,36,1.310443,-1.206534,No,0.0218,0.9782,0.7961,0.2038,129219906,GO:0001708 [Name: cell fate specification]; GO...,T-box transcription factor TBX3-like
1428,N5.HOG0073157,Loss more likely in non-orb-weavers,34,-3.186391,1.988284,Yes,0.9576,0.0424,0.1151,0.8846,NaN,NaN,NaN
1429,N5.HOG0073180,Loss more likely in non-orb-weavers,32,-1.038058,0.724732,No,0.9929,0.0071,0.1394,0.8606,129221249,-,cuticle protein 38-like
1430,N5.HOG0073251,Loss more likely in non-orb-weavers,33,-3.434295,1.656853,Yes,0.9866,0.0134,0.2178,0.7822,NaN,NaN,NaN


In [85]:
%autoreload 2
perm_df = get_ptep_description(perm_locs_filtered)
perm_df

,HOG,Result,Occupancy,Log odds ratio of loss,Log odds ratio of duplication,LOR outside permuted avg thresholds,"P-value, loss more likely in orb-weavers","P-value, loss more likely in non-orb-weavers",P-value duplication more likely in fg,P-value duplication more likely in bg,U. diversus gene ID,U. diversus GO terms,U. diversus gene description,P. tepidariorum best BLAST hit for the HOG
0,N5.HOG0001041,Loss more likely in orb-weavers,46,2.288042,-0.281735,No,0.0038,0.9962,0.5328,0.4672,NaN,NaN,NaN,gastrula zinc finger protein XlCGF7.1
1,N5.HOG0001609,Loss more likely in non-orb-weavers,49,-2.841587,2.391485,No,0.9605,0.0395,0.0747,0.9253,NaN,NaN,NaN,speckle-type POZ protein
2,N5.HOG0001627,"Loss more likely in orb-weavers, duplication m...",33,2.048043,-4.020457,No for loss; yes for duplication,0.0449,0.9551,0.9745,0.0253,NaN,NaN,NaN,speckle-type POZ protein
3,N5.HOG0002031,Loss more likely in non-orb-weavers,46,-3.823601,1.854638,Yes,0.9996,0.0004,0.3311,0.6686,129233975,GO:0008010 [Name: structural constituent of ch...,cuticle protein 10.9-like,cuticle protein 10.9
4,N5.HOG0002072,Loss more likely in orb-weavers,52,3.473733,-0.762726,Yes,0.0108,0.9892,0.7989,0.2011,129217362,GO:0008010 [Name: structural constituent of ch...,proteoglycan 4-like,uncharacterized protein
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1427,N5.HOG0073114,Loss more likely in orb-weavers,36,1.310443,-1.206534,No,0.0218,0.9782,0.7961,0.2038,129219906,GO:0001708 [Name: cell fate specification]; GO...,T-box transcription factor TBX3-like,T-box transcription factor TBX3
1428,N5.HOG0073157,Loss more likely in non-orb-weavers,34,-3.186391,1.988284,Yes,0.9576,0.0424,0.1151,0.8846,NaN,NaN,NaN,uncharacterized protein
1429,N5.HOG0073180,Loss more likely in non-orb-weavers,32,-1.038058,0.724732,No,0.9929,0.0071,0.1394,0.8606,129221249,-,cuticle protein 38-like,cuticle protein 63-like
1430,N5.HOG0073251,Loss more likely in non-orb-weavers,33,-3.434295,1.656853,Yes,0.9866,0.0134,0.2178,0.7822,NaN,NaN,NaN,forkhead box protein K1 isoform X2


In [86]:
perm_df.to_excel(os.path.join(repo_root, 'results/Supplementary_Table_5_OddsRatioTest_hits.xlsx'), index=False)

In [103]:
previous_interesting = pd.read_csv(os.path.join(repo_root, 'data/interesting_hits.txt'), header=None, names=['HOG'])
previous_interesting_df = perm_df[perm_df['HOG'].isin(previous_interesting['HOG'])]
previous_interesting_df.to_excel(os.path.join(repo_root, 'results/Supplementary_Table_6_OddsRatioTest_interesting_hits.xlsx'), index=False)
previous_interesting_df

,HOG,Result,Occupancy,Log odds ratio of loss,Log odds ratio of duplication,LOR outside permuted avg thresholds,"P-value, loss more likely in orb-weavers","P-value, loss more likely in non-orb-weavers",P-value duplication more likely in fg,P-value duplication more likely in bg,U. diversus gene ID,U. diversus GO terms,U. diversus gene description,P. tepidariorum best BLAST hit for the HOG
43,N5.HOG0007453,Duplication more likely in non-orb-weavers,50,1.611461,-1.686717,No,0.1080,0.8920,0.9501,0.0499,129216356,GO:0043005 [Name: neuron projection],lachesin-like,lachesin
145,N5.HOG0016850,Loss more likely in non-orb-weavers,52,-1.470832,-0.617585,No,0.9868,0.0132,0.6464,0.3534,129234104,GO:0006468 [Name: protein phosphorylation]; GO...,"cGMP-dependent protein kinase, isozyme 2 forms...",cGMP-dependent protein kinase 1 isoform X4
167,N5.HOG0018539,Loss more likely in non-orb-weavers,84,-1.626481,0.765561,No,0.9864,0.0136,0.1582,0.8418,129230744,GO:0005737 [Name: cytoplasm]; GO:0097320 [Name...,protein kinase C and casein kinase substrate i...,protein kinase C and casein kinase substrate i...
190,N5.HOG0019920,Loss more likely in non-orb-weavers,81,-3.668799,0.925429,Yes,0.9766,0.0234,0.0952,0.9048,129222487,GO:0005515 [Name: protein binding]; GO:0007052...,mitogen-activated protein kinase-binding prote...,mitogen-activated protein kinase-binding prote...
224,N5.HOG0022330,Loss more likely in non-orb-weavers,50,-2.749680,2.013628,No,0.9837,0.0163,0.0719,0.9281,129217545,GO:0046777 [Name: protein autophosphorylation]...,dual specificity tyrosine-phosphorylation-regu...,dual specificity tyrosine-phosphorylation-regu...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1264,N5.HOG0068366,Loss more likely in non-orb-weavers,79,-2.661433,0.541285,No,0.9598,0.0402,0.2621,0.7379,129218629,GO:0007169 [Name: transmembrane receptor prote...,docking protein 3-like,docking protein 1
1274,N5.HOG0068715,Loss more likely in non-orb-weavers,83,-2.309868,0.931903,No,0.9958,0.0042,0.1128,0.8872,129232140,GO:0004930 [Name: G-protein coupled receptor a...,probable G-protein coupled receptor AH9.1,probable G-protein coupled receptor AH9.1
1287,N5.HOG0068937,Loss more likely in orb-weavers,78,1.869858,0.195768,No,0.0052,0.9948,0.4507,0.5493,129225023,GO:0007218 [Name: neuropeptide signaling pathway],FMRFamide propeptide,myomodulin neuropeptides
1303,N5.HOG0069291,Duplication more likely in orb-weavers,81,-1.061098,1.991745,No,0.9257,0.0743,0.0183,0.9817,129231096,GO:0046580 [Name: negative regulation of Ras p...,sprouty signaling antagonist,protein sprouty


## Spider accessions and BUSCO scores  

In [136]:
buscos = pd.read_csv(f'{data}/orthorun_list_fams_busco90.csv')
accessions = pd.read_csv(f'{data}/txptome_accessions.csv')
accessions.drop(columns=['Infraorder', 'Family', 'Web_Type', 'Cribellum'], inplace=True)

buscos_accessions = pd.merge(buscos, accessions, on='Species', how='left')

pd.set_option('display.max_columns', None, 'display.max_rows', None)

buscos_accessions.drop_duplicates(inplace=True)
buscos_accessions.dropna(how='any', inplace=True)
buscos_accessions.reset_index(drop=True, inplace=True)
# Collapse rows: group by all columns except 'Accession', join 'Accession' values as comma-separated string
buscos_accessions.rename(columns={'Single': 'Single_copy_BUSCOs', 'Duplicated': 'Duplicated_BUSCOs', 'Total': 'Total_BUSCOs'}, inplace=True)
buscos_accessions


,Species,Infraorder,Family,Single_copy_BUSCOs,Duplicated_BUSCOs,Total_BUSCOs,Accession
0,Argiope_bruennichi,Araneomorphae,Araneidae,2354,374,2728,IBFJ01
1,Argiope_aetheroides,Araneomorphae,Araneidae,2344,375,2719,IASB01
2,Caerostris_darwini,Araneomorphae,Araneidae,2337,358,2695,ICOF01
3,Argiope_keyserlingi,Araneomorphae,Araneidae,2324,394,2718,IAET01
4,Eriophora_pustulosa,Araneomorphae,Araneidae,2321,421,2742,IBIK01
5,Caerostris_extrusa,Araneomorphae,Araneidae,2319,336,2655,ICOE01
6,Backobourkia_brouni,Araneomorphae,Araneidae,2313,342,2655,ICCG01
7,Acrosomoides_sp_IDV7426,Araneomorphae,Araneidae,2306,338,2644,ICOJ01
8,Poecilopachys_australasia,Araneomorphae,Araneidae,2306,388,2694,ICKB01
9,Dolophones_sp_IDV6683,Araneomorphae,Araneidae,2304,440,2744,IBZG01


In [137]:
buscos_accessions.to_excel(f'{results}/Supplementary_Table_1_SpiderAccessions_BUSCOs.xlsx', index=False)


In [138]:
# Count rows with NaN in the 'Accession' column before collapsing
num_nan_accession = buscos_accessions['Accession'].isna().sum()
print(f"Rows with NaN in 'Accession': {num_nan_accession}")

Rows with NaN in 'Accession': 0


## N5 HOGs

this is just to make the format match the others

In [139]:
n5_hogs = pd.read_csv(f'{data}/N5.tsv', sep='\t')
n5_hogs.to_excel(f'{results}/Supplementary_Table_2_Entelegyne_HOGs.xlsx', index=False)

/var/folders/h8/wkfyd5dj3nnd_smftqfplclh0000gn/T/ipykernel_73349/2718854308.py:1: DtypeWarning: Columns (3,4,5,6,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,36,37,39,40,41,42,43,44,45,46,47,48,49,51,52,53,54,55,56,57,58,59,60,61,62,63,64,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94) have mixed types. Specify dtype option on import or set low_memory=False.
  n5_hogs = pd.read_csv(f'{data}/N5.tsv', sep='\t')


## PhyloGLM results

In [112]:
phyloglm = pd.read_csv(f'{results}/phyloglm/phyloglm_continuous_qvals.csv', index_col="HOG")
phyloglm

,error,coef_.Intercept._Estimate,coef_.Intercept._StdErr,coef_.Intercept._z.value,coef_.Intercept._p.value,coef_orb_weavingTRUE_Estimate,coef_orb_weavingTRUE_StdErr,coef_orb_weavingTRUE_z.value,coef_orb_weavingTRUE_p.value,qvalue
HOG,,,,,,,,,,
N5.HOG0000042,NaN,-1.939954,1.802701,-1.076137,0.281866,0.175860,1.186312,0.148240,0.882153,0.874180
N5.HOG0000052,NaN,-0.919683,0.884749,-1.039485,0.298579,0.546297,0.554324,0.985521,0.324368,0.644266
N5.HOG0000147,NaN,-1.238526,0.753393,-1.643930,0.100191,0.027575,0.516438,0.053394,0.957418,0.882163
N5.HOG0000196,NaN,-0.247401,0.584944,-0.422948,0.672333,-1.524352,0.964151,-1.581030,0.113871,0.368647
N5.HOG0000276,NaN,-1.854115,2.445372,-0.758214,0.448323,-0.954439,1.626479,-0.586813,0.557329,0.793873
...,...,...,...,...,...,...,...,...,...,...
N5.HOG0073724,NaN,-1.382774,0.613846,-2.252641,0.024282,0.583460,0.384139,1.518876,0.128794,0.394173
N5.HOG0073740,NaN,-0.704353,0.379234,-1.857304,0.063268,0.081796,0.255739,0.319840,0.749090,0.855650
N5.HOG0073766,NaN,-1.891086,1.078839,-1.752890,0.079621,1.658120,0.753386,2.200890,0.027744,0.155473


In [113]:
phyloglm_sig = phyloglm[phyloglm['qvalue'] < 0.05].drop(columns=[
    'error', 'coef_.Intercept._Estimate', 'coef_.Intercept._StdErr', 'coef_.Intercept._z.value', 'coef_.Intercept._p.value'])
phyloglm_sig.rename(columns={
    'coef_orb_weavingTRUE_Estimate': 'Orb-weaving coefficient estimate',
    'coef_.coef_orb_weavingTRUE_StdErr': 'Orb-weaving coefficient standard error',
    'coef_.coef_orb_weavingTRUE_z.value': 'Orb-weaving coefficient z-value',
    'coef_.coef_orb_weavingTRUE_p.value': 'Orb-weaving coefficient p-value'
    }, inplace=True)

In [114]:
phyloglm_locs = convert_hogs_to_locs(phyloglm_sig, os.path.join(repo_root, "data/N5.tsv"), one_random_gene=True)
phyloglm_locs_fltrd = phyloglm_locs.drop(columns=['udiv_genes', 'dmel_orthologs', 'txpt']).drop_duplicates().reset_index(drop=True)
phyloglm_locs_fltrd = move_column(phyloglm_locs_fltrd, 'HOG', 0)
phyloglm_locs_fltrd.rename(columns={
    'LOC': 'U. diversus gene ID',
    'GO_terms': 'U. diversus GO terms',
    'Description': 'U. diversus gene description'
    }, inplace=True)
phyloglm_locs_fltrd

Processing HOGs: 100%|██████████| 897/897 [00:00<00:00, 49709.20it/s]


,HOG,Orb-weaving coefficient estimate,coef_orb_weavingTRUE_StdErr,coef_orb_weavingTRUE_z.value,coef_orb_weavingTRUE_p.value,qvalue,U. diversus gene ID,U. diversus GO terms,U. diversus gene description
0,N5.HOG0002054,0.700399,0.194774,3.595947,0.000323,0.009889,129227793,GO:0008010 [Name: structural constituent of ch...,adult-specific rigid cuticular protein 15.7-like
1,N5.HOG0002298,1.402103,0.407769,3.438473,0.000585,0.014122,129220903,GO:0006508 [Name: proteolysis]; GO:0004222 [Na...,uncharacterized LOC129220903
2,N5.HOG0004006,1.422759,0.409604,3.473501,0.000514,0.012997,NaN,NaN,NaN
3,N5.HOG0006044,-0.772249,0.262935,-2.937031,0.003314,0.042794,129218517,-,arylsulfatase B-like
4,N5.HOG0007085,0.607532,0.199406,3.046710,0.002314,0.033634,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
892,N5.HOG0072549,1.488362,0.465060,3.200366,0.001373,0.024361,NaN,NaN,NaN
893,N5.HOG0072608,1.125855,0.391306,2.877171,0.004013,0.048074,129234448,GO:0005634 [Name: nucleus]; GO:0015630 [Name: ...,tektin-5-like
894,N5.HOG0072851,3.356720,0.711405,4.718436,0.000002,0.000500,NaN,NaN,NaN
895,N5.HOG0073069,1.156261,0.371091,3.115843,0.001834,0.029350,129231753,-,uncharacterized LOC129231753


In [119]:
phyloglm_df = get_ptep_description(phyloglm_locs_fltrd)
phyloglm_df.to_excel(os.path.join(repo_root, 'results/Supplementary_Table_7_PhyloGLM_hits.xlsx'), index=False)
phyloglm_df

,HOG,Orb-weaving coefficient estimate,coef_orb_weavingTRUE_StdErr,coef_orb_weavingTRUE_z.value,coef_orb_weavingTRUE_p.value,qvalue,U. diversus gene ID,U. diversus GO terms,U. diversus gene description,P. tepidariorum best BLAST hit for the HOG
0,N5.HOG0002054,0.700399,0.194774,3.595947,0.000323,0.009889,129227793,GO:0008010 [Name: structural constituent of ch...,adult-specific rigid cuticular protein 15.7-like,adult-specific rigid cuticular protein 15.7-li...
1,N5.HOG0002298,1.402103,0.407769,3.438473,0.000585,0.014122,129220903,GO:0006508 [Name: proteolysis]; GO:0004222 [Na...,uncharacterized LOC129220903,astacin-like metalloprotease toxin 5 isoform X1
2,N5.HOG0004006,1.422759,0.409604,3.473501,0.000514,0.012997,NaN,NaN,NaN,uncharacterized protein
3,N5.HOG0006044,-0.772249,0.262935,-2.937031,0.003314,0.042794,129218517,-,arylsulfatase B-like,arylsulfatase B-like
4,N5.HOG0007085,0.607532,0.199406,3.046710,0.002314,0.033634,NaN,NaN,NaN,cathepsin L-like peptidase
...,...,...,...,...,...,...,...,...,...,...
892,N5.HOG0072549,1.488362,0.465060,3.200366,0.001373,0.024361,NaN,NaN,NaN,obscurin
893,N5.HOG0072608,1.125855,0.391306,2.877171,0.004013,0.048074,129234448,GO:0005634 [Name: nucleus]; GO:0015630 [Name: ...,tektin-5-like,tektin-5
894,N5.HOG0072851,3.356720,0.711405,4.718436,0.000002,0.000500,NaN,NaN,NaN,NO_HIT
895,N5.HOG0073069,1.156261,0.371091,3.115843,0.001834,0.029350,129231753,-,uncharacterized LOC129231753,NO_HIT
